In [1]:
import pandas as pd
import numpy as np

from snowflake.snowpark import Session
from credentials import params

from snowflake.ml.registry import Registry
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from pydantic import BaseModel
import re

In [2]:
session = Session.builder.configs(params).create()
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

In [3]:
train_set = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET").to_pandas()
y_train = train_set['PRICE_IN_LAKHS']
X_train = train_set.drop('PRICE_IN_LAKHS', axis = 1)

In [4]:
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)

# Preprocessing

In [5]:
model = registry.get_model("PREPROCESSING_PIPELINE")
model.show_versions()

,created_on,name,aliases,comment,database_name,schema_name,model_name,is_default_version,functions,metadata,user_data,model_attributes,size,environment,runnable_in,inference_services
0,2026-08-31 09:47:33.846000-07:00,RUN_20260831_094722,"[""DEFAULT"",""FIRST"",""LAST""]",None,HOUSING_PRICE_PROJECT,ML_LAYER,PREPROCESSING_PIPELINE,true,"[""TRANSFORM""]",{},{},"{""framework"":""snowml"",""client"":""snowflake-ml-p...",43452,"{""default"":{""python_version"":""3.12"",""snowflake...","[""WAREHOUSE"",""SNOWPARK_CONTAINER_SERVICES""]",[]


In [6]:
preproc_model = registry \
    .get_model("PREPROCESSING_PIPELINE") \
    .version("DEFAULT")

In [7]:
X_train_prep = preproc_model.run(
    X_train,
    function_name="transform"
)

In [ ]:
#list(X_train_prep.columns)

In [40]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   CITY                        30000 non-null  object 
 1   LOCALITY                    30000 non-null  object 
 2   LOCALITY_TIER               30000 non-null  object 
 3   PROPERTY_TYPE               30000 non-null  object 
 4   BALCONIES                   30000 non-null  int8   
 5   CARPET_AREA                 30000 non-null  int16  
 6   FLOOR_NUMBER                30000 non-null  int8   
 7   TOTAL_FLOORS                30000 non-null  int8   
 8   FLOOR_CATEGORY              30000 non-null  object 
 9   FACING                      30000 non-null  object 
 10  FURNISHING_STATUS           30000 non-null  object 
 11  PROPERTY_AGE                30000 non-null  int8   
 12  PARKING_SPACES              30000 non-null  int8   
 13  SECURITY_SCORE              300

# Prediction

In [9]:
predict_model = registry \
    .get_model("HOUSING_PRICE_XGBOOST") \
    .version("DEFAULT")

In [10]:
import re
def sanitize_column_name(column):
    column = column.replace('+', 'plus')
    column = re.sub(r'[()"]', '', column)
    column = re.sub(r'[\s-]', '_', column)                   
    column = re.sub(r'[^a-zA-Z0-9_]', '', column)
    return column.upper()

In [11]:
X_train_prep = X_train_prep.rename(columns = {col : sanitize_column_name(col) for col in X_train_prep.columns})

In [ ]:
list(X_train_prep.columns)

In [13]:
predictions = predict_model.run(
    X_train_prep,
    function_name="predict"
)

In [14]:
rmse = mean_squared_error(
    y_true = y_train,
    y_pred = predictions
)

mae = mean_absolute_error(
    y_true = y_train,
    y_pred = predictions
)

r2 = r2_score(
    y_true = y_train,
    y_pred = predictions
)

In [15]:
print(rmse)
print(mae)
print(r2)

1642.772272653006
30.015946047815955
0.849919632608203


In [28]:
X_train_prep.iloc[[0],:]

,BALCONIES,CARPET_AREA,FLOOR_NUMBER,TOTAL_FLOORS,PROPERTY_AGE,PARKING_SPACES,SECURITY_SCORE,GYM_AVAILABLE,SWIMMING_POOL,POWER_BACKUP,...,FACING_NORTH_WEST,FACING_SOUTH,FACING_SOUTH_EAST,FACING_SOUTH_WEST,FACING_WEST,FURNISHING_STATUS_FULLY_FURNISHED,FURNISHING_STATUS_SEMI_FURNISHED,FURNISHING_STATUS_UNFURNISHED,TRANSACTION_TYPE_NEW,TRANSACTION_TYPE_RESALE
0,1.0,1550.0,13.0,17.0,9.0,2.0,6.2,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [38]:
i = 3
pred = predict_model.run(
    X_train_prep.iloc[[i],:],
    function_name="predict"
)
print(pred.output_feature_0[0])
print(y_train[i])

130.29489
136.31


241.68121
241.43


# FastAPI

### Outside Functions

In [41]:
class Property(BaseModel):
    CITY: str
    LOCALITY: str
    LOCALITY_TIER: str
    PROPERTY_TYPE: str
    BALCONIES: int
    CARPET_AREA: int
    FLOOR_NUMBER: int
    TOTAL_FLOORS: int
    FLOOR_CATEGORY: str
    FACING: str
    FURNISHING_STATUS: str
    PROPERTY_AGE: int
    PARKING_SPACES: int
    SECURITY_SCORE: float
    GYM_AVAILABLE: int
    SWIMMING_POOL: int
    POWER_BACKUP: int
    LIFT_AVAILABLE: int
    MAINTENANCE_FEE_MONTHLY: int
    DISTANCE_TO_CITY_CENTER_KM: float
    DISTANCE_TO_METRO_KM: float
    NEARBY_SCHOOLS: int
    NEARBY_HOSPITALS: int
    TRANSACTION_TYPE: str

### endpoint function

In [85]:
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)

def endpoint(record: pd.DataFrame):

    record_p = preprocessing(record, registry)
    pred = prediction(record_p, registry)
    return pred

### preprocessing function

In [54]:
def preprocessing(record: pd.DataFrame, registry):
    
    preproc_model = registry \
        .get_model("PREPROCESSING_PIPELINE") \
        .version("DEFAULT")
    
    record_p = preproc_model.run(
        record,
        function_name="transform"
    )

    return record_p.rename(columns = {col : sanitize_column_name(col) for col in record_p.columns})

### column name cleaning function

In [55]:
def sanitize_column_name(column):
    
    column = column.replace('+', 'plus')
    column = re.sub(r'[()"]', '', column)
    column = re.sub(r'[\s-]', '_', column)                   
    column = re.sub(r'[^a-zA-Z0-9_]', '', column)
    return column.upper()

### prediction function

In [59]:
def prediction(record_p: pd.DataFrame, registry):
    
    predict_model = registry \
        .get_model("HOUSING_PRICE_XGBOOST") \
        .version("DEFAULT")
    
    prediction = predict_model.run(
        record_p,
        function_name="predict"
    )

    return prediction.output_feature_0[0]

In [86]:
a = endpoint(X_train.iloc[[3],:])
print(a)

130.29489
<class 'numpy.float32'>


# HTTP Client to query API

In [63]:
import requests

In [100]:
response = requests.post(
    "http://127.0.0.1:8000/pred",
    json = X_train.iloc[3,:].to_dict()
)

In [99]:
response.json()

{'price_estim': 130.29489135742188}

In [69]:
X_train.iloc[3,:].to_dict()

{'CITY': 'Mumbai',
 'LOCALITY': 'Bandra',
 'LOCALITY_TIER': 'Premium',
 'PROPERTY_TYPE': 'Apartment',
 'BALCONIES': 2,
 'CARPET_AREA': 770,
 'FLOOR_NUMBER': 19,
 'TOTAL_FLOORS': 23,
 'FLOOR_CATEGORY': 'High (10-19)',
 'FACING': 'East',
 'FURNISHING_STATUS': 'Semi-Furnished',
 'PROPERTY_AGE': 4,
 'PARKING_SPACES': 1,
 'SECURITY_SCORE': 5.6,
 'GYM_AVAILABLE': 0,
 'SWIMMING_POOL': 0,
 'POWER_BACKUP': 1,
 'LIFT_AVAILABLE': 1,
 'MAINTENANCE_FEE_MONTHLY': 2942,
 'DISTANCE_TO_CITY_CENTER_KM': 8.5,
 'DISTANCE_TO_METRO_KM': 3.4,
 'NEARBY_SCHOOLS': 8,
 'NEARBY_HOSPITALS': 2,
 'TRANSACTION_TYPE': 'Resale'}

{'CITY': 'Mumbai',
 'LOCALITY': 'Bandra',
 'LOCALITY_TIER': 'Premium',
 'PROPERTY_TYPE': 'Apartment',
 'BALCONIES': 2,
 'CARPET_AREA': 770,
 'FLOOR_NUMBER': 19,
 'TOTAL_FLOORS': 23,
 'FLOOR_CATEGORY': 'High (10-19)',
 'FACING': 'East',
 'FURNISHING_STATUS': 'Semi-Furnished',
 'PROPERTY_AGE': 4,
 'PARKING_SPACES': 1,
 'SECURITY_SCORE': 5.6,
 'GYM_AVAILABLE': 0,
 'SWIMMING_POOL': 0,
 'POWER_BACKUP': 1,
 'LIFT_AVAILABLE': 1,
 'MAINTENANCE_FEE_MONTHLY': 2942,
 'DISTANCE_TO_CITY_CENTER_KM': 8.5,
 'DISTANCE_TO_METRO_KM': 3.4,
 'NEARBY_SCHOOLS': 8,
 'NEARBY_HOSPITALS': 2,
 'TRANSACTION_TYPE': 'Resale'}

In [ ]:
#1. Request parsing
#2. Data preprocessing
#3. Model inference

In [97]:
print("[1/3] Request parsing   ", end="\t")
1+1
print("COMPLETED!")
print("[2/3] Data preprocessing", end="\t")
1+1
print("COMPLETED!")
print("[3/3] Model inference   ", end="\t")
1+1
print("COMPLETED!")

[1/3] Request parsing   	COMPLETED!
[2/3] Data preprocessing	COMPLETED!
[3/3] Model inference   	COMPLETED!
